# Train Math CNN on Google Colab (GPU)

Trains the **17-class** model (`0-9` + `+ - * / = ( )`) with **both** datasets merged.

**Setup**
1. Colab: **Runtime → Change runtime type → T4 GPU**
2. On your PC: zip `datasets/math_external` → `math_external.zip`
3. Run all cells below

**After training:** download outputs into your repo:
- `backend/models/math_model.pth`
- `backend/models/math_labels.json`

In [ ]:
import subprocess
import sys

!nvidia-smi

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "torch", "torchvision", "scikit-learn", "matplotlib", "pillow", "numpy",
])

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU detected. Set Runtime -> T4 GPU, then restart runtime.")

In [ ]:
import os
from pathlib import Path

REPO = "https://github.com/ayesha-saboora/SpatialAICanvas-Final-Grad-Proj.git"
ROOT = Path("/content/SpatialAICanvas-Final-Grad-Proj")

if not ROOT.exists():
    !git clone {REPO} {ROOT}

os.chdir(ROOT)
print("Repo ready at", ROOT)

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

ROOT = Path("/content/SpatialAICanvas-Final-Grad-Proj")
dest = ROOT / "datasets" / "math_external"
dest.mkdir(parents=True, exist_ok=True)

print("Upload math_external.zip from your PC...")
uploaded = files.upload()

zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name, "r") as zf:
    zf.extractall(dest)

required = [
    dest / "Handwritten math symbols dataset",
    dest / "math numbers dataset (0-9) only",
]
for p in required:
  print("OK" if p.is_dir() else "MISSING", p.name)

In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = Path("/content/SpatialAICanvas-Final-Grad-Proj")
script = ROOT / "ml" / "train_math.py"

cmd = [
    sys.executable,
    str(script),
    "--epochs", "15",
    "--batch-size", "256",
    "--max-per-class", "3000",
]
print("Running:", " ".join(cmd))
subprocess.check_call(cmd, cwd=ROOT)

In [ ]:
import json
from pathlib import Path
from google.colab import files

ROOT = Path("/content/SpatialAICanvas-Final-Grad-Proj")
metrics = json.loads((ROOT / "ml" / "math_metrics.json").read_text())
print("Best test accuracy:", metrics.get("best_test_accuracy"))

for p in [
    ROOT / "backend/models/math_model.pth",
    ROOT / "backend/models/math_labels.json",
    ROOT / "ml/math_metrics.json",
    ROOT / "ml/math_confusion_matrix.png",
]:
    print("Downloading", p.name)
    files.download(str(p))